In [10]:
import pandas as pd

# Cleaning the data
### 1. Details Df
##### We are ignoring the columns that contains an high number of empty values and columns that are not important for the sake of the analysis

In [11]:
dfDetails = pd.read_csv("../data/raw/details.csv")

dfDetails.drop_duplicates()
dfDetails = dfDetails.drop(columns=["title_japanese",
                                    "url",
                                    "image_url",
                                    "start_date",
                                    "end_date",
                                    "synopsis",
                                    "studios",
                                    "demographics",
                                    "season",
                                    "licensors",
                                    "explicit_genres",
                                    "streaming"],
                           errors = "ignore")

dfDetails["type"] = dfDetails["type"].fillna("Unknown")
dfDetails["rating"] = dfDetails["rating"].fillna("Unknown")

dfDetails["score"] = dfDetails["score"].fillna(0)
dfDetails["scored_by"] = dfDetails["scored_by"].fillna(0)
dfDetails["rank"] = dfDetails["rank"].fillna(0)

dfDetails["episodes"] = dfDetails["episodes"].fillna(0)
dfDetails["episodes"] = dfDetails["episodes"].astype(int)

dfDetails["year"] = dfDetails["year"].fillna(0)
dfDetails["year"] = dfDetails["year"].astype(int)

dfDetails

,mal_id,title,type,status,score,scored_by,rank,popularity,members,favorites,genres,themes,source,rating,episodes,year,producers
0,59356,-Socket-,Movie,Finished Airing,0.00,0.0,17086.0,22507,195,0,['Comedy'],[],Original,G - All Ages,1,0,['Nagoya Zokei University']
1,56036,......,Music,Finished Airing,6.53,503.0,0.0,15004,941,2,"['Horror', 'Supernatural']",['Music'],Original,PG-13 - Teens 13 or older,1,0,[]
2,2928,.hack//G.U. Returner,OVA,Finished Airing,6.65,9745.0,6366.0,5056,22525,31,"['Adventure', 'Drama', 'Fantasy']",['Video Game'],Game,PG-13 - Teens 13 or older,1,0,"['Bandai Visual', 'CyberConnect2']"
3,3269,.hack//G.U. Trilogy,Movie,Finished Airing,7.06,15373.0,4194.0,4215,34264,104,"['Action', 'Fantasy']",['Video Game'],Game,PG-13 - Teens 13 or older,1,0,['Bandai Visual']
4,4469,.hack//G.U. Trilogy: Parody Mode,Special,Finished Airing,6.35,4317.0,8182.0,6696,11135,10,"['Comedy', 'Fantasy', 'Sci-Fi']","['Parody', 'Video Game']",Game,PG-13 - Teens 13 or older,1,0,['Bandai Visual']
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28950,59421,Zutaboro Reijou wa Ane no Moto Konyakusha ni D...,TV,Finished Airing,7.37,15624.0,2562.0,3621,46157,182,"['Drama', 'Romance']",[],Light novel,PG-13 - Teens 13 or older,12,2025,"['Studio Pierrot', 'Mainichi Broadcasting Syst..."
28951,31245,Zutto Mae kara Suki deshita. Kokuhaku Jikkou I...,Movie,Finished Airing,7.20,104106.0,3494.0,1077,249631,682,['Romance'],['School'],Music,PG-13 - Teens 13 or older,1,0,"['Aniplex', 'Dentsu', 'Kadokawa Shoten', 'Movi..."
28952,36305,Zutto Mae kara Suki deshita. Kokuhaku Jikkou I...,Special,Finished Airing,7.17,10038.0,3664.0,4680,27334,34,['Romance'],['School'],Music,PG - Children,1,0,['Aniplex']
28953,34895,Zutto Suki Datta,OVA,Finished Airing,5.68,1887.0,0.0,8916,5525,11,"['Drama', 'Hentai']",['School'],Manga,Rx - Hentai,2,0,"['Queen Bee', 'Mediabank']"


### 2. Stats df
##### Same goes for the stats, ignoring columns that can be obtained with simple calculations or obtained in other ways

##### Decided to keep the score_x_votes to analyze better in the future§

In [12]:
dfStats = pd.read_csv("../data/raw/stats.csv")

dfStats = dfStats.drop(columns=dfStats.filter(regex="percentage$").columns)
dfStats.drop_duplicates()
dfStats

,mal_id,watching,completed,on_hold,dropped,plan_to_watch,total,score_1_votes,score_2_votes,score_3_votes,score_4_votes,score_5_votes,score_6_votes,score_7_votes,score_8_votes,score_9_votes,score_10_votes
0,59356,7,146,4,20,20,197,2.0,0.0,3.0,6.0,25.0,33.0,19.0,2.0,0.0,1.0
1,56036,21,770,8,29,113,941,5.0,6.0,8.0,14.0,50.0,138.0,144.0,81.0,17.0,40.0
2,2928,451,14953,302,349,6472,22527,101.0,93.0,164.0,457.0,1184.0,2054.0,2709.0,1500.0,875.0,608.0
3,3269,726,22790,452,537,9762,34267,120.0,156.0,260.0,560.0,1270.0,2457.0,4157.0,3075.0,1919.0,1400.0
4,4469,241,6918,182,266,3528,11135,83.0,104.0,182.0,292.0,683.0,888.0,871.0,592.0,308.0,315.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28950,59421,11349,17353,574,1440,15729,46445,54.0,40.0,95.0,255.0,854.0,2303.0,5545.0,3919.0,1598.0,1235.0
28951,31245,10332,140676,2989,1416,94249,249662,357.0,484.0,1011.0,2269.0,7199.0,15972.0,34370.0,24387.0,10480.0,7589.0
28952,36305,928,16119,388,243,9656,27334,48.0,28.0,78.0,182.0,773.0,1839.0,3258.0,2009.0,887.0,936.0
28953,34895,896,2387,375,356,1512,5526,116.0,80.0,102.0,172.0,337.0,437.0,280.0,170.0,68.0,125.0


### Decided to merge dfDetails and dfStats to make a complete dataset of the anime

##### Freeing Memory by setting the previous df to None

In [13]:
animeDf = dfDetails.merge(dfStats, how = "left", on = "mal_id")
dfStats = None
dfDetails = None

##### Renaming

In [14]:
animeDf = animeDf.rename(columns={"total": "total_users"})
animeDf

,mal_id,title,type,status,score,scored_by,rank,popularity,members,favorites,...,score_1_votes,score_2_votes,score_3_votes,score_4_votes,score_5_votes,score_6_votes,score_7_votes,score_8_votes,score_9_votes,score_10_votes
0,59356,-Socket-,Movie,Finished Airing,0.00,0.0,17086.0,22507,195,0,...,2.0,0.0,3.0,6.0,25.0,33.0,19.0,2.0,0.0,1.0
1,56036,......,Music,Finished Airing,6.53,503.0,0.0,15004,941,2,...,5.0,6.0,8.0,14.0,50.0,138.0,144.0,81.0,17.0,40.0
2,2928,.hack//G.U. Returner,OVA,Finished Airing,6.65,9745.0,6366.0,5056,22525,31,...,101.0,93.0,164.0,457.0,1184.0,2054.0,2709.0,1500.0,875.0,608.0
3,3269,.hack//G.U. Trilogy,Movie,Finished Airing,7.06,15373.0,4194.0,4215,34264,104,...,120.0,156.0,260.0,560.0,1270.0,2457.0,4157.0,3075.0,1919.0,1400.0
4,4469,.hack//G.U. Trilogy: Parody Mode,Special,Finished Airing,6.35,4317.0,8182.0,6696,11135,10,...,83.0,104.0,182.0,292.0,683.0,888.0,871.0,592.0,308.0,315.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28950,59421,Zutaboro Reijou wa Ane no Moto Konyakusha ni D...,TV,Finished Airing,7.37,15624.0,2562.0,3621,46157,182,...,54.0,40.0,95.0,255.0,854.0,2303.0,5545.0,3919.0,1598.0,1235.0
28951,31245,Zutto Mae kara Suki deshita. Kokuhaku Jikkou I...,Movie,Finished Airing,7.20,104106.0,3494.0,1077,249631,682,...,357.0,484.0,1011.0,2269.0,7199.0,15972.0,34370.0,24387.0,10480.0,7589.0
28952,36305,Zutto Mae kara Suki deshita. Kokuhaku Jikkou I...,Special,Finished Airing,7.17,10038.0,3664.0,4680,27334,34,...,48.0,28.0,78.0,182.0,773.0,1839.0,3258.0,2009.0,887.0,936.0
28953,34895,Zutto Suki Datta,OVA,Finished Airing,5.68,1887.0,0.0,8916,5525,11,...,116.0,80.0,102.0,172.0,337.0,437.0,280.0,170.0,68.0,125.0


##### Saving the cleaned df

In [15]:
animeDf.to_csv("../data/cleaned/anime.csv", index = False)
animeDf = None

### 3. Recommendation df
##### Cleaning the recommendation by removing duplicates and eventual self recommendation

In [16]:
dfRecs = pd.read_csv("../data/raw/recommendations.csv")

dfRecs = dfRecs[dfRecs["mal_id"] != dfRecs["recommendation_mal_id"]]
dfRecs.drop_duplicates()

,mal_id,recommendation_mal_id
0,3269,317
1,3269,6922
2,3269,299
3,3269,3446
4,3269,5681
...,...,...
105244,31245,1689
105245,31245,35434
105246,31245,31798
105247,31245,21995


In [17]:
dfRecs = dfRecs.rename(columns={"recommendation_mal_id": "recommended_anime_id"})

dfRecs

,mal_id,recommended_anime_id
0,3269,317
1,3269,6922
2,3269,299
3,3269,3446
4,3269,5681
...,...,...
105244,31245,1689
105245,31245,35434
105246,31245,31798
105247,31245,21995


##### Saving the normalized df

In [18]:
dfRecs.to_csv("../data/cleaned/recommendation.csv", index = False)
dfRecs = None